In [4]:
#%%shell
!pip3 install vl-convert-python

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 19.6 MB 1.2 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [11]:
import pandas as pd
import altair as alt
import json
import vl_convert as vlc
import time
from PIL import Image

# df=pd.DataFrame({'Org': ['school1','school1','school1','school1', 'school2', 'school2', 'school2', 'school2', 'school3', 'school3', 'school3', 'school3'],
#                  'variable': ['rose', 'dropped', 'maintained', 'failed','rose', 'dropped', 'maintained', 'failed', 'rose', 'dropped', 'maintained', 'failed'],
#                 'numbers': [25,35,25,45,25,25,25,25,40,15,35,30], 
#                  'position': [12.5, 42.5, 72.5, 107.5, 12.5, 37.5,62.5, 87.5, 12.5, 42.5, 72.5, 107.5]})

df=pd.DataFrame({'Org': ['school1','school1','school1','school1', 'school2', 'school2', 'school2', 'school2', 'school3', 'school3', 'school3', 'school3'],
                 'variable': ['rose', 'dropped', 'maintained', 'failed','rose', 'dropped', 'maintained', 'failed', 'rose', 'dropped', 'maintained', 'failed'],
                'numbers': [14, 13, 13, 60, 22, 11, 20, 47, 24, 13, 22, 41]})

#---1. increase distance between bars -- now they are overlapping. Done
# Change figure size OR/AND
# Change bar size
bar_size = 50

#2, scale bar below bars from 0 to 100.
# Adjust numbers

#3. names of legend (rose etc. should be changed to other names) changeable
# Since Altair legends reflect the content of the data. The cleanest way to change the values they show is to change the values within the dataset
# Also set color for values here

## colors
# purple #6e05ab
# light blue #2462bf
# lightgrey
# blue-purple: #3f3999
value_map = {
    'dropped' : ['test_dropped_alt','#2462bf'],
    'failed' : ['failed_alt','#3f3999'],
    'maintained' : ['maintained_alt','lightgrey'],
    'rose' : ['rose_alt','#6e05ab'],
}
df['variable'] = df['variable'].apply(lambda x: value_map[x][0])
domain = [i[0] for i in value_map.values()]
range = [i[1] for i in value_map.values()]

#4. larger font size numbers on scale below the bars
#5. larger fontsize for legend text and larger legend in general

# Set font sizes
axis_labelFontSize = 16
title_FontSize = 16
text_FontSize = 12
legend_labelFontSize = 16
legend_titleFontSize = 16

#6. saving the figure as jpg and eps
#---7. correct numbers: first bar: 14, 13, 13, 60 second bar: 22, 11, 20, 47, third bar: 24, 13, 22, 41, plot
# adjusted to show the correct numbers with correct section size of bars

### Direction of bars
direction = 'horizantal'  # or 'vertical'

if direction == 'horizantal':
    ascending=[True, True]

    x_bar_str = 'sum(numbers)'
    x_text_str = 'position'
    y_bar_str = 'Org'
    y_text_str = 'Org'

else:
    ascending=[True, False]
    x_bar_str = 'Org'
    y_bar_str = 'sum(numbers)'
    x_text_str = 'Org'
    y_text_str = 'position'

# Need to calculate correct postion to show text (middle of bars)
df.sort_values(by=['Org','variable'],ascending=ascending,inplace=True)
df['cul'] = df.groupby(['Org'])['numbers'].cumsum()
df['position'] = df.apply(lambda row: row['cul'] - row['numbers'] // 2,axis=1)

### Plot
bars=alt.Chart().mark_bar(stroke='black', strokeWidth=2, opacity=0.6, size=bar_size).encode(
    x=alt.X(x_bar_str, title='Number of Students'),
    y=alt.Y(y_bar_str, title=None),
    color=alt.Color('variable', title=None,
                    scale=alt.Scale(domain=domain,
                                    range=range),
                    legend=alt.Legend(title='variable', orient = 'right',labelFontSize=legend_labelFontSize,titleFontSize=legend_titleFontSize))
)

text=alt.Chart().mark_text(align='center', baseline='middle', size=text_FontSize).encode(
    y=alt.Y(y_text_str, title=None),
    x=x_text_str,
    text='numbers')

chart_out = alt.layer(bars, text, data=df).properties(
    width=300,
    height=240
).configure_axis(
    labelFontSize = axis_labelFontSize,
    titleFontSize = title_FontSize,
)

chart_out

alt.LayerChart(...)

In [6]:
# Save as jpg and eps
# No direct way to save an altair chart to desired format.
# json -> png -> jpg & eps
chart_out.save('temp_json.json')
with open('temp_json.json') as f:
    temp_json = json.load(f)

time.sleep(5)
png_data = vlc.vegalite_to_png(vl_spec=temp_json, scale=2)
with open('chart.png', "wb") as f:
    f.write(png_data)

im = Image.open(r'chart.png')
rgb_im = im.convert('RGB')
rgb_im.save(r'chart.jpg')
rgb_im.save('chart.eps')